In [1]:
# importing everything
import numpy as np
import pandas as pd
import re

from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import classification_report, f1_score
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

import os

In [2]:
print("=== Dataset Overview ===")
print(f"Training samples : {train.shape[0]}")
print(f"Test samples     : {test.shape[0]}")
print(f"Total features   : {train.shape[1]}")
print(f"\nColumn names:\n{list(train.columns)}")
print(f"\nData types:\n{train.dtypes}")
plt.figure(figsize=(8, 4))
label_counts = train["label"].value_counts().sort_index()
bars = plt.bar(label_counts.index, label_counts.values, 
               color=["#4C72B0","#DD8452","#55A868","#C44E52"])
plt.title("Class Distribution (Label Counts)", fontsize=14)
plt.xlabel("Label")
plt.ylabel("Count")
for bar, count in zip(bars, label_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
             f'{count:,}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

print("\nClass percentages:")
print((label_counts / len(train) * 100).round(2).astype(str) + "%")

=== Dataset Overview ===


NameError: name 'train' is not defined

In [ ]:
# 3. Missing values analysis 
print("=== Missing Values ===")
missing = train.isnull().sum()
missing_pct = (missing / len(train) * 100).round(2)
missing_df = pd.DataFrame({"Missing Count": missing, 
                            "Missing %": missing_pct})
print(missing_df[missing_df["Missing Count"] > 0])

# also check for hidden "none" strings
print("\n=== Hidden None Strings ===")
for col in numeric_cols:
    none_count = (train[col] == "none").sum()
    if none_count > 0:
        print(f"{col}: {none_count} 'none' strings found")

In [ ]:
# 4. Numeric feature distributions
pure_numeric_cols = ["emoticon_1", "emoticon_2", "emoticon_3",
                     "upvote", "downvote", "if_1", "if_2"]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(pure_numeric_cols):
    axes[i].hist(train[col].replace("none", 0).astype(float), 
                 bins=30, color="#4C72B0", edgecolor="white")
    axes[i].set_title(f"{col} distribution")
    axes[i].set_xlabel("Value")
    axes[i].set_ylabel("Count")

axes[-1].set_visible(False)
plt.suptitle("Numeric Feature Distributions", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 5. Categorical feature analysis
cat_cols = ["race", "religion", "gender", "disability"]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for i, col in enumerate(cat_cols):
    counts = train[col].replace("none", "unknown").fillna("unknown").value_counts()
    axes[i].bar(counts.index, counts.values, color="#55A868", edgecolor="white")
    axes[i].set_title(f"{col} distribution")
    axes[i].set_xlabel("Category")
    axes[i].set_ylabel("Count")
    axes[i].tick_params(axis='x', rotation=30)

plt.suptitle("Categorical Feature Distributions", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 6. Comment length analysis
# do longer comments belong to certain categories?

train["comment_length"] = train["comment"].apply(lambda x: len(str(x)))
train["word_count"]     = train["comment"].apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label in sorted(train["label"].unique()):
    subset = train[train["label"] == label]["comment_length"]
    axes[0].hist(subset, bins=40, alpha=0.5, label=f"Class {label}")
axes[0].set_title("Comment Length by Class")
axes[0].set_xlabel("Character Count")
axes[0].legend()

for label in sorted(train["label"].unique()):
    subset = train[train["label"] == label]["word_count"]
    axes[1].hist(subset, bins=40, alpha=0.5, label=f"Class {label}")
axes[1].set_title("Word Count by Class")
axes[1].set_xlabel("Word Count")
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nAverage comment length per class:")
print(train.groupby("label")["comment_length"].mean().round(1))
print("\nAverage word count per class:")
print(train.groupby("label")["word_count"].mean().round(1))

In [ ]:
# 7. Upvote/Downvote vs Label
# do certain classes get more engagement?

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

train.boxplot(column="upvote", by="label", ax=axes[0])
axes[0].set_title("Upvotes by Class")
axes[0].set_xlabel("Label")
axes[0].set_ylabel("Upvotes")

train.boxplot(column="downvote", by="label", ax=axes[1])
axes[1].set_title("Downvotes by Class")
axes[1].set_xlabel("Label")
axes[1].set_ylabel("Downvotes")

plt.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
# Model Comparison

results = {
    "Logistic Regression"  : f1_score(y_val, lr_preds,       average="macro"),
    "Random Forest"        : f1_score(y_val, rf_preds,        average="macro"),
    "XGBoost"              : f1_score(y_val, preds_xgb,       average="macro"),
    "RF Balanced"          : f1_score(y_val, preds_balanced,  average="macro"),
    "Combined (LR+RF+XGB)" : f1_score(y_val, preds_3,        average="macro"),
}

print("=== Model Comparison (Macro F1) ===\n")
for model, score in sorted(results.items(), key=lambda x: x[1], reverse=True):
    bar = "█" * int(score * 50)
    print(f"{model:<25} {score:.5f}  {bar}")

# bar chart
plt.figure(figsize=(10, 5))
plt.bar(results.keys(), results.values(), color="#4C72B0", edgecolor="white")
plt.axhline(y=0.80, color="red", linestyle="--", label="0.80 target")
plt.title("Model Comparison — Macro F1 Score")
plt.ylabel("Macro F1")
plt.xticks(rotation=15)
plt.legend()
plt.ylim(0.70, 0.85)
plt.tight_layout()
plt.show()

print("\nBest model:", max(results, key=results.get))
print("Best score:", round(max(results.values()), 5))

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

train = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/train.csv")
test  = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/test.csv")

print("Train shape:", train.shape)
print("Test shape: ", test.shape)
print()
print(train.head())

In [ ]:
# if one class has way more rows than others, the dataset is "imbalanced"
print("Label distribution:")
print(train["label"].value_counts())
print()

# check for missing val
print("Missing values in train:")
print(train.isnull().sum())
print()

# look at numeric col
numeric_cols = [
    "emoticon_1", "emoticon_2", "emoticon_3",
    "upvote", "downvote",
    "if_1", "if_2",
    "race", "religion", "gender", "disability"
]
print("Numeric feature stats:")
print(train[numeric_cols].describe())

In [ ]:
# we will check this
def clean_text_v2(text):
    text = str(text).lower()
    
    text = re.sub(r"http\S+", " ", text)        # remove URLs
    text = re.sub(r"@\w+", " ", text)            # remove @mentions  
    text = re.sub(r"#\w+", " ", text)            # remove hashtags
    text = re.sub(r"[^a-zA-Z\s]", " ", text)    # remove special chars
    text = re.sub(r"\s+", " ", text)             # remove extra spaces
    text = text.strip()
    return text

train["clean_comment"] = train["comment"].apply(clean_text_v2)
test["clean_comment"]  = test["comment"].apply(clean_text_v2)

# let's check what cleaned comments look like
print(train[["comment", "clean_comment"]].head(3))

In [ ]:
# add to train and test before building feature matrix
def add_features(df):
    df = df.copy()
    
    # text length features
    df["char_count"]      = df["comment"].apply(lambda x: len(str(x)))
    df["word_count"]      = df["comment"].apply(lambda x: len(str(x).split()))
    df["unique_words"]    = df["comment"].apply(lambda x: len(set(str(x).lower().split())))
    
    # punctuation features — strong signal for toxic/emotional comments
    df["exclaim_count"]   = df["comment"].apply(lambda x: str(x).count("!"))
    df["question_count"]  = df["comment"].apply(lambda x: str(x).count("?"))
    df["caps_count"]      = df["comment"].apply(lambda x: sum(1 for c in str(x) if c.isupper()))
    
    # ratio features
    df["caps_ratio"]      = df["caps_count"] / (df["char_count"] + 1)
    df["unique_ratio"]    = df["unique_words"] / (df["word_count"] + 1)
    
    return df

train = add_features(train)
test  = add_features(test)

# add new columns to your numeric list
extra_cols = ["char_count", "word_count", "unique_words",
              "exclaim_count", "question_count", 
              "caps_count", "caps_ratio", "unique_ratio"]

print("New features added:", extra_cols)
print(train[extra_cols].head())

pure_numeric_cols = ["emoticon_1", "emoticon_2", "emoticon_3",
                     "upvote", "downvote", "if_1", "if_2"] + extra_cols

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=15000,      
    stop_words="english",
    ngram_range=(1, 3),
    sublinear_tf=True,       
    min_df=2,                
    strip_accents="unicode"
)

X_train_text = vectorizer.fit_transform(train["clean_comment"])
X_test_text  = vectorizer.transform(test["clean_comment"])

print("Text feature shape:", X_train_text.shape)

In [ ]:
# Step 1: separate columns into two groups

# numeric col
pure_numeric_cols = ["emoticon_1", "emoticon_2", "emoticon_3",
                     "upvote", "downvote", "if_1", "if_2"]

# categorical text col
categorical_cols = ["race", "religion", "gender", "disability"]

# Step 2: clean and scale the numeric columns
X_train_num = train[pure_numeric_cols].replace("none", np.nan).fillna(0).astype(float)
X_test_num  = test[pure_numeric_cols].replace("none", np.nan).fillna(0).astype(float)

scaler = StandardScaler()
X_train_num_scaled = scaler.fit_transform(X_train_num)  
X_test_num_scaled  = scaler.transform(X_test_num)

# Step 3: encode the categorical columns
train_cat = train[categorical_cols].replace("none", np.nan).fillna("missing").astype(str)
test_cat  = test[categorical_cols].replace("none", np.nan).fillna("missing").astype(str)

train_encoded = pd.get_dummies(train_cat)
test_encoded  = pd.get_dummies(test_cat)

train_encoded, test_encoded = train_encoded.align(test_encoded, join="left", axis=1, fill_value=0)

print("Numeric features shape:", X_train_num_scaled.shape)
print("Categorical features shape:", train_encoded.shape)

# Step 4: convert everything to sparse and combine
X_train_numeric_sparse = csr_matrix(
    np.hstack([X_train_num_scaled, train_encoded.values])
)
X_test_numeric_sparse = csr_matrix(
    np.hstack([X_test_num_scaled, test_encoded.values])
)

print("Final numeric+categorical sparse shape:", X_train_numeric_sparse.shape)

In [ ]:
X_train_all = hstack([X_train_text, X_train_numeric_sparse])
X_test_all  = hstack([X_test_text,  X_test_numeric_sparse])

y = train["label"]

print("Final training feature shape:", X_train_all.shape)

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_all, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training size:  ", X_tr.shape[0])
print("Validation size:", X_val.shape[0])

In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_tr, y_tr)

lr_preds = lr_model.predict(X_val)

print("=== Logistic Regression Results ===")
print(classification_report(y_val, lr_preds))

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,    
    random_state=42,
    n_jobs=-1           
)
rf_model.fit(X_tr, y_tr)

rf_preds = rf_model.predict(X_val)

print("=== Random Forest Results ===")
print(classification_report(y_val, rf_preds))

# compare F1 scores
lr_f1 = f1_score(y_val, lr_preds, average="weighted")
rf_f1 = f1_score(y_val, rf_preds, average="weighted")
print(f"\nLogistic Regression F1: {lr_f1:.4f}")
print(f"Random Forest F1:       {rf_f1:.4f}")
print(f"\nBest model: {'Random Forest' if rf_f1 > lr_f1 else 'Logistic Regression'}")

In [ ]:
rf_balanced = RandomForestClassifier(
    n_estimators=200,           
    class_weight="balanced",  
    random_state=42,
    n_jobs=-1
)
rf_balanced.fit(X_tr, y_tr)
preds_balanced = rf_balanced.predict(X_val)

print(classification_report(y_val, preds_balanced))

In [ ]:
# train both models separately first
lr = LogisticRegression(max_iter=1000, class_weight="balanced", C=5, random_state=42)
rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1)

lr.fit(X_tr, y_tr)
rf.fit(X_tr, y_tr)

# get probability predictions from each
lr_proba = lr.predict_proba(X_val)
rf_proba = rf.predict_proba(X_val)

# average them 
combined_proba = (0.4 * lr_proba) + (0.6 * rf_proba)
combined_preds = np.argmax(combined_proba, axis=1)

print("=== Combined Model ===")
print(classification_report(y_val, combined_preds))

In [ ]:
# short comments behave differently than long ones
train["comment_length"]    = train["clean_comment"].apply(len)
train["word_count"]        = train["clean_comment"].apply(lambda x: len(x.split()))
train["avg_word_length"]   = train["clean_comment"].apply(
    lambda x: np.mean([len(w) for w in x.split()]) if len(x.split()) > 0 else 0
)

test["comment_length"]     = test["clean_comment"].apply(len)
test["word_count"]         = test["clean_comment"].apply(lambda x: len(x.split()))
test["avg_word_length"]    = test["clean_comment"].apply(
    lambda x: np.mean([len(w) for w in x.split()]) if len(x.split()) > 0 else 0
)

# then add these to numeric_cols before scaling
pure_numeric_cols = ["emoticon_1", "emoticon_2", "emoticon_3",
                     "upvote", "downvote", "if_1", "if_2",
                     "comment_length", "word_count", "avg_word_length"] 

In [ ]:
# try different blend ratios and see which gives best validation score
for lr_weight in [0.3, 0.4, 0.5, 0.6, 0.7]:
    rf_weight = 1 - lr_weight
    combined = (lr_weight * lr_proba) + (rf_weight * rf_proba)
    preds = np.argmax(combined, axis=1)
    score = f1_score(y_val, preds, average="macro")
    print(f"LR: {lr_weight:.1f} | RF: {rf_weight:.1f} | Macro F1: {score:.5f}")

In [ ]:
lr_stronger = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    C=10,               # increased from 5
    solver="saga",      
    random_state=42
)
lr_stronger.fit(X_tr, y_tr)
preds_lr_strong = lr_stronger.predict(X_val)
print(classification_report(y_val, preds_lr_strong))

In [ ]:
xgb_tuned = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)
xgb_tuned.fit(X_tr, y_tr)
preds_xgb_tuned = xgb_tuned.predict(X_val)

print("=== Tuned XGBoost ===")
print(classification_report(y_val, preds_xgb_tuned))
print("Macro F1:", f1_score(y_val, preds_xgb_tuned, average="macro"))

In [ ]:
# use probabilities from tuned xgb now
xgb_proba = xgb_tuned.predict_proba(X_val)

best_score = 0
best_weights = None

for lr_w in [0.2, 0.3, 0.4, 0.5]:
    for rf_w in [0.2, 0.3, 0.4, 0.5]:
        xgb_w = round(1 - lr_w - rf_w, 1)
        if xgb_w <= 0:
            continue
        
        combined = (lr_w * lr_proba) + (rf_w * rf_proba) + (xgb_w * xgb_proba)
        preds = np.argmax(combined, axis=1)
        score = f1_score(y_val, preds, average="macro")
        
        if score > best_score:
            best_score = score
            best_weights = (lr_w, rf_w, xgb_w)

print(f"Best weights → LR: {best_weights[0]} | RF: {best_weights[1]} | XGB: {best_weights[2]}")
print(f"Best macro F1: {best_score:.5f}")

In [ ]:
lr_final = LogisticRegression(
    max_iter=1000, class_weight="balanced", C=5, random_state=42
)
rf_final = RandomForestClassifier(
    n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1
)
xgb_final = XGBClassifier(
    n_estimators=500,        
    learning_rate=0.05,      
    max_depth=7,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

lr_final.fit(X_train_all, y)
rf_final.fit(X_train_all, y)
xgb_final.fit(X_train_all, y)

lr_proba_final  = lr_final.predict_proba(X_test_all)
rf_proba_final  = rf_final.predict_proba(X_test_all)
xgb_proba_final = xgb_final.predict_proba(X_test_all)

combined_final = (0.35 * lr_proba_final) + (0.35 * rf_proba_final) + (0.30 * xgb_proba_final)
predictions = np.argmax(combined_final, axis=1)

# Submission
submission = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/Sample.csv")
submission["label"] = predictions
submission.to_csv("submission.csv", index=False)

print("\nSubmission saved!")
print(submission.head(10))
print(f"\nPrediction counts:\n{pd.Series(predictions).value_counts()}")